# Stage 2 — Assignment DQN

The Assignment DQN decides what to do with each incoming order:

| Action | Name         | Behaviour                          | Reward    |
|--------|--------------|------------------------------------|-----------|
| 0      | Accept       | Nav to pickup → nav to dropoff     | nav + 100 |
| 1      | Decline-idle | Idle briefly, wait for next order  | −20~−17.5 |
| 2      | GoCharge     | Nav to charger, charge to 100%     | −25       |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import matplotlib
matplotlib.use('Agg')

from envs.assign_env import WarehouseStage2, PICKUP_POINTS
from agents.dqn import AssignmentDQN
from utils.plotting import render_assign_grid, plot_assign_history, decision_heatmap
from training.train_assign import load_nav_policy, train_stage2, evaluate_stage2, DEVICE, CKPT_DIR

print(f'Device: {DEVICE}')

In [ ]:
render_assign_grid(title='Stage 2 — Fixed 10×10 Warehouse')

In [ ]:
nav_model = load_nav_policy()

In [ ]:
assign_dqn, history = train_stage2(
    nav_model     = nav_model,
    episodes      = 8_000,
    n_orders      = 5,
    gamma         = 0.99,
    lr            = 3e-4,
    batch         = 128,
    eps_start     = 1.0,
    eps_decay     = 0.9993,
    eps_min       = 0.05,
    warmup        = 300,
    verbose_every = 500,
)

In [ ]:
plot_assign_history(history, out_dir=CKPT_DIR)

In [ ]:
eval_metrics = evaluate_stage2(nav_model, assign_dqn, n_eval=200)

In [ ]:
decision_heatmap(assign_dqn, DEVICE, out_dir=CKPT_DIR)

In [ ]:
import json, numpy as np

torch.save(assign_dqn.state_dict(), f'{CKPT_DIR}/assign_dqn.pt')
for key, arr in history.items():
    np.save(f'{CKPT_DIR}/assign_{key}.npy', arr)
with open(f'{CKPT_DIR}/assign_eval.json', 'w') as f:
    json.dump({k: float(v) for k, v in eval_metrics.items()}, f, indent=2)
print('Saved to checkpoints/')